In [ ]:
!pip install earthpy gdal --quiet
!pip install torch torchvision segmentation-models-pytorch --quiet

In [ ]:
from osgeo import gdal, gdal_array
import os
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from tqdm import tqdm
import random
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from skimage import io
import joblib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from torch import amp
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import math
from PIL import Image
from torch.optim.lr_scheduler import StepLR
import copy
import cv2
import segmentation_models_pytorch as smp
import pickle



mask_path_train="/kaggle/input/resized-clean-splitted-augmented-cloud-data/cleaned-data/train/masks"
data_path_train ="/kaggle/input/resized-clean-splitted-augmented-cloud-data/cleaned-data/train/images"

mask_path_test="/kaggle/input/resized-clean-splitted-augmented-cloud-data/cleaned-data/test/masks"
data_path_test ="/kaggle/input/resized-clean-splitted-augmented-cloud-data/cleaned-data/test/images"

mask_path_val="/kaggle/input/resized-clean-splitted-augmented-cloud-data/cleaned-data/val/masks"
data_path_val ="/kaggle/input/resized-clean-splitted-augmented-cloud-data/cleaned-data/val/images"

pca = joblib.load('/kaggle/input/pca/other/default/1/pca_model.joblib')
scaler = joblib.load('/kaggle/input/pca/other/default/1/scaler_model.joblib')


In [ ]:
train_data= os.listdir(data_path_train)
test_data= os.listdir(data_path_test)
val_data= os.listdir(data_path_val)

print(len(train_data))
print(len(test_data))
print(len(val_data))

***Data Augmentation***

In [ ]:
def add_gaussian_noise(img, mean=0.0, std=0.2):
    img = img.astype(np.float32)
    img = (img-img.min()) / (img.max()-img.min())  # Convert from [0, 255] to [0, 1]
    noise = np.random.normal(mean, std, img.shape).astype(np.float32)
    noisy_img = img + noise
    noisy_img = np.clip(noisy_img, 0., 1.)
    return (noisy_img*((img.max()-img.min())))+img.min()

In [ ]:
def random_augment(mask,image):
    op = random.choice(['flip_horizontal', 'flip_vertical', 'rotate_90', 'rotate_180', 'rotate_270', 'noise'])
    
    if op == 'flip_horizontal':
        image = cv2.flip(image, 1)   # Horizontal flip
        mask = cv2.flip(mask, 1)   # Horizontal flip
    elif op == 'flip_vertical':
        image = cv2.flip(image, 0)   # Vertical flip
        mask = cv2.flip(mask, 0)   # Vertical flip
    elif op == 'rotate_90':
        image = cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)
        mask = cv2.rotate(mask, cv2.ROTATE_90_CLOCKWISE)
    elif op == 'rotate_180':
        image = cv2.rotate(image, cv2.ROTATE_180)
        mask = cv2.rotate(mask, cv2.ROTATE_180)
    elif op == 'rotate_270':
        image = cv2.rotate(image, cv2.ROTATE_90_COUNTERCLOCKWISE)
        mask = cv2.rotate(mask, cv2.ROTATE_90_COUNTERCLOCKWISE)
    elif op == 'noise':
        image = add_gaussian_noise(image)
    return mask,image


In [ ]:
img_count = 0
for i,img in tqdm(enumerate(train_data),desc="reading data and augmenting ..."):
    img_pca=apply_pca(img,True)
    img_pca = cv2.resize(img_pca, (256, 256), interpolation=cv2.INTER_LINEAR)

    new_mask_path = os.path.join(mask_path, img)  
    mask_data = gdal.Open(new_mask_path, gdal.GA_ReadOnly)
    mask = mask_data.GetRasterBand(1).ReadAsArray()
    mask = cv2.resize(mask, (256, 256), interpolation=cv2.INTER_NEAREST)

    # Save image
    img_pca_pil = Image.fromarray(img_pca)
    img_pca_pil.save(f"cleaned-data/train/images/{img}")
    # Save the corresponding mask
    mask_pil = Image.fromarray(mask.astype(np.uint8))
    mask_pil.save(f"cleaned-data/train/masks/{img}")
    # augment data
 
    new_mask,new_img=random_augment(mask,img_pca)
    new_mask = cv2.resize(new_mask, (256, 256), interpolation=cv2.INTER_NEAREST)
    new_img = cv2.resize(new_img, (256, 256), interpolation=cv2.INTER_LINEAR)

    # Save the augmented image and mask
    img_pil = Image.fromarray(new_img)
    mask_pil = Image.fromarray(new_mask.astype(np.uint8))
    # save the augmented data
    img_pil.save(f"cleaned-data/train/images/aug_{img_count}.tif")
    mask_pil.save(f"cleaned-data/train/masks/aug_{img_count}.tif")
    img_count += 1
    


***Dataset***

In [ ]:
class CloudMaskDataset(Dataset):
    def __init__(self,data_path,mask_path,image_files):
        self.image_paths = [os.path.join(data_path, fname) for fname in image_files]
        self.mask_paths = [os.path.join(mask_path, fname) for fname in image_files]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # read img
        img_data= gdal.Open(self.image_paths[idx], gdal.GA_ReadOnly)
        img_band = img_data.GetRasterBand(1)
        img = img_band.ReadAsArray()
        
        # read mask
        mask_data= gdal.Open(self.mask_paths[idx], gdal.GA_ReadOnly)
        band = mask_data.GetRasterBand(1)
        mask = band.ReadAsArray()
        
        # # Normalize to [0, 1]
        img = (img.astype(np.float16)-img.min()) / (img.max()-img.min())
        
        # Expand channel dimension: [H, W] → [1, H, W]
        img = np.expand_dims(img, axis=0)
        mask = np.expand_dims(mask, axis=0)

        return torch.from_numpy(img), torch.from_numpy(mask)


***Training the Model***

In [ ]:
grad_scaler = GradScaler()  
def train(model, loader, optimizer):
    model.train()
    epoch_loss = 0.0

    for imgs, masks in tqdm(loader, desc="training loop"):
        torch.cuda.empty_cache()

        imgs = imgs.to(torch.float16).to(device)   # you can still use float16 explicitly
        masks = masks.to(torch.float16).to(device)

        optimizer.zero_grad()

        with autocast():  # enable automatic mixed precision
            outputs = model(imgs)
            loss = calc_loss(outputs, masks)

        # backward pass with scaled loss
        grad_scaler.scale(loss).backward()

        # gradient clipping to avoid NaNs
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # optimizer step
        grad_scaler.step(optimizer)
        grad_scaler.update()

        epoch_loss += loss.item()

    return epoch_loss / len(loader)

***Dice Loss and Dice Score***

In [ ]:
def dice_loss(pred, target, smooth=1e-6):
    pred = F.sigmoid(pred)
    pred = pred.contiguous()
    target = target.contiguous()
    intersection = (pred * target).sum(dim=2).sum(dim=2)
    loss = (1 - ((2. * intersection + smooth) / (pred.sum(dim=2).sum(dim=2) + target.sum(dim=2).sum(dim=2) + smooth)))

    return loss.mean()

def calc_loss(pred, target, bce_weight=0.5):
    bce = F.binary_cross_entropy_with_logits(pred, target)
    dice = dice_loss(pred, target)
    loss = bce * bce_weight + dice * (1 - bce_weight)
    return loss

def dice_score(pred, target, smooth=1e-7):
    pred = F.sigmoid(pred)
    pred = (pred > 0.5)
    pred = pred.contiguous()
    target = target.contiguous()
    intersection = (pred * target).sum(dim=2).sum(dim=2)
    dice = ((2. * intersection + smooth) / (pred.sum(dim=2).sum(dim=2) + target.sum(dim=2).sum(dim=2) + smooth))

    return dice.mean()


***Evaluation of the model***

In [ ]:
def evaluate(model, loader,debug):
    model.eval()
    epoch_loss = 0
    epoch_dice = 0
    with torch.no_grad():
        for i,(imgs, masks )in tqdm(enumerate(loader),desc="testing loop"):
            torch.cuda.empty_cache()
            imgs = imgs.to(torch.float32).to(device)
            masks = masks.to(torch.float32).to(device)
            outputs = model(imgs)
            if masks is not None:
                loss=calc_loss(outputs, masks)
                dice =dice_score(outputs, masks)
                epoch_loss += loss.item()
                epoch_dice += dice.item()
            if debug and i%4==0  :
                    preds = torch.sigmoid(outputs)  # (B, 1, H, W)
                    preds = (preds > 0.5).float()
                    
                    pred_mask = preds[0, 0].cpu().detach().numpy()  # shape: (H, W)
                    true_mask = masks[0, 0].cpu().detach().numpy()
                    input_image = imgs[0,0].cpu().detach().numpy()
                    
                    plt.figure(figsize=(12, 4))
                    
                    plt.subplot(1, 3, 1)
                    plt.title("Input Image")
                    plt.imshow(input_image, cmap='gray')
                    plt.axis('off')
                    
                    plt.subplot(1, 3, 2)
                    plt.title("Ground Truth Mask")
                    plt.imshow(true_mask, cmap='gray',vmin=0,vmax=1)
                    plt.axis('off')
                    
                    plt.subplot(1, 3, 3)
                    plt.title("Predicted Mask")
                    plt.imshow(pred_mask, cmap='gray',vmin=0,vmax=1)
                    plt.axis('off')
                    
                    plt.tight_layout()
                    plt.show()
                

    avg_loss = epoch_loss / len(loader)
    avg_dice = epoch_dice / len(loader)
    return avg_loss, avg_dice


***Creation of Loaders***

In [ ]:
# Create dataset and dataloader for train_data
train_dataset = CloudMaskDataset(data_path_train,mask_path_train,train_data)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True,num_workers=4)

# Create dataset and dataloader for validation
val_dataset = CloudMaskDataset(data_path_val,mask_path_val,val_data)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=True,num_workers=4)

# Create dataset and dataloader tset
test_dataset = CloudMaskDataset(data_path_test,mask_path_test,test_data)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=True,num_workers=4)




***Initializaion of Model***

In [ ]:
# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize model and move to device
model = smp.Unet(
    encoder_name='efficientnet-b0',     
    encoder_weights='imagenet',     # Pretrained weights
    in_channels=1,                  # Input channels (e.g., RGB)
    classes=1,                      # Output channels (binary segmentation)
    activation=None                 # No activation (logits output)
)
model = torch.nn.DataParallel(model)
model = model.to(device)

# Define optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)



***Training Loop***

In [ ]:
epochs=25
best_model_wts = copy.deepcopy(model.state_dict())
best_dice = 0
for epoch in range(epochs):
    train_loss = train(model, train_loader, optimizer)
    val_loss, val_dice = evaluate(model, val_loader,False)
    if val_dice > best_dice:
        best_dice = val_dice
        best_model_wts = copy.deepcopy(model.state_dict())
    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Dice: {val_dice:.4f}")


In [ ]:
#Save the model 
model.load_state_dict(best_model_wts)
torch.save(model, 'unet_eff.pth')
print("model saved successfully...")

In [ ]:
# Save the model using pickle
model_to_save = model.module if isinstance(model, torch.nn.DataParallel) else model
with open("'unet_eff.pkl", "wb") as f:
    pickle.dump(model_to_save, f)
print("Model saved as a pickle file successfully...")

In [ ]:

val_loss, val_dice = evaluate(model, val_loader,True)
print(f" Val Loss: {val_loss:.4f}, Val Dice: {val_dice:.4f}")

test_loss, test_dice = evaluate(model, test_loader,True)
print(f" test Loss: {test_loss:.4f}, test Dice: {test_dice:.4f}")


***Data Cleaning***

In [ ]:
# def clean_data(model,data_path,mask_path,data,debug):
#     data_to_be_deleted=[]
#     count=0
#     for file_name in tqdm(data,desc="cleaning loop"):
#         img_data_path =  os.path.join(data_path, file_name)
#         img_data= gdal.Open(img_data_path, gdal.GA_ReadOnly)
#         img_band = img_data.GetRasterBand(1)
#         img = img_band.ReadAsArray()
#         img = (img.astype(np.float16)-img.min()) / (img.max()-img.min())
#         img = np.expand_dims(img, axis=0)
#         img = np.expand_dims(img, axis=0)
#         img=torch.from_numpy(img)
#         img = img.to(torch.float32).to(device)
        
#         mask_data_path =  os.path.join(mask_path, file_name)
#         mask_data= gdal.Open(mask_data_path, gdal.GA_ReadOnly)
#         mask_band = mask_data.GetRasterBand(1)
#         mask = mask_band.ReadAsArray()
#         mask = np.expand_dims(mask, axis=0)
#         mask = np.expand_dims(mask, axis=0)
#         mask=torch.from_numpy(mask)
#         mask = mask.to(torch.float32).to(device)
        
#         torch.cuda.empty_cache()
#         with torch.no_grad():
#             pred = torch.sigmoid(model(img))               
#             pred_label = (pred > 0.5).float()               
#             confident_mask = (pred > 0.8) | (pred < 0.2)      
#             mismatch = pred_label != mask          
#             suspicious_pixels = mismatch & confident_mask    
            
#             suspicious_ratio = suspicious_pixels[0].sum() / suspicious_pixels[0].numel()
#             if suspicious_ratio > 0.5:
#                 data_to_be_deleted.append(file_name)
#                 if debug:
#                     pred_mask = pred_label[0, 0].cpu().detach().numpy()  # shape: (H, W)
#                     true_mask = mask[0, 0].cpu().detach().numpy()
#                     input_image = img[0,0].cpu().detach().numpy()
                    
#                     # Plot the input, true mask, and predicted mask
#                     plt.figure(figsize=(12, 4))
                    
#                     plt.subplot(1, 3, 1)
#                     plt.title("Input Image")
#                     plt.imshow(input_image, cmap='gray')
#                     plt.axis('off')
                    
#                     plt.subplot(1, 3, 2)
#                     plt.title("Ground Truth Mask")
#                     plt.imshow(true_mask, cmap='gray',vmin=0,vmax=1)
#                     plt.axis('off')
                    
#                     plt.subplot(1, 3, 3)
#                     plt.title("Predicted Mask")
#                     plt.imshow(pred_mask, cmap='gray',vmin=0,vmax=1)
#                     plt.axis('off')
                    
#                     plt.tight_layout()
#                     plt.show()
                    
#     return data_to_be_deleted
                

In [ ]:
# train_data_to_be_deleted=clean_data(model2,data_path_train,mask_path_train,train_data,True)
# print(len(train_data))
# print(len(train_data_to_be_deleted))

In [ ]:
# val_data_to_be_deleted=clean_data(model,data_path_val,mask_path_val,val_data,True)
# print(len(val_data))
# print(len(val_data_to_be_deleted))

In [ ]:
# test_data_to_be_deleted=clean_data(model,data_path_test,mask_path_test,test_data,True)
# print(len(test_data))
# print(len(test_data_to_be_deleted))

In [ ]:
# clean_train_data =np.array([x for x in train_data if x not in train_data_to_be_deleted])
# print("train len ",len(clean_train_data))
# clean_val_data =np.array([x for x in val_data if x not in val_data_to_be_deleted])
# print("val len ",len(clean_val_data))
# clean_test_data =np.array([x for x in test_data if x not in test_data_to_be_deleted])
# print("test len ",len(clean_test_data))

In [ ]:
# np.save('clean_train_data.npy',clean_train_data)
# np.save('clean_val_data.npy',clean_val_data)
# np.save('clean_test_data.npy',clean_test_data)